# Proyecto — Data Stream Processor

## Contexto (extremadamente importante): 

Uno de las mayores virtudes de un repositorio en github es la poder volver sobre los cambios hechos. Uno puede volver sbre algún `commit` he iniciar el proceso desde ese punto. 

En otro ejemplo parecido, al crear una lista en python, dicha lista se modifica al agregar o borrar elementos y regresar a un estado anterior de la lista no es tan sencillo. La idea de este proyecto es poder emular dicho proceso y realizar una especie de lista con memoria para poder llevar algunos registros de manera adecuada.


### Objetivo

Construya un pequeño sistema para recibir y procesar registros de datos utilizando las clases `ArrayStack` y `ArrayQueue` proporcionadas por el curso. La idea central es poder llevar la información en orden para llevar los cambios de los registros de manera adecuada.

En el método `__init__` debe aparecer tres atributos:
- **Queue:** registros que han llegado pero todavía no han sido procesados.
- **Stack:** historial de cambios realizados, para poder deshacer los cambios más recientes.
- **Lista:** como se encuentran los registros actualmente

Las clases `ArrayStack` y `ArrayQueue` ya están implementadas. **No debe implementarlas nuevamente ni modificarlas.**

In [26]:
from goodrich.ch06.array_stack import ArrayStack
from goodrich.ch06.array_queue import ArrayQueue

In [27]:
class DataStreamProcessor:
    def __init__(self):
        
        self.queue = ArrayQueue() 
        
        self.stack = ArrayStack() 
        
        self.lista = []

## 1. Registro de datos

Cada registro es una tupla de tres elementos:

```python
(sensor, variable, value)
```

Ejemplos:

```python
("S01", "temperature", 23.5)
("S02", "temperature", 25.1)
("S01", "humidity", 61.2)
```

Una combinación única `(sensor, variable)` identifica un dato dentro del estado actual.

In [28]:
def recibir_registro(self, registro):
    """Recibe una tupla (sensor, variable, valor) y la pone al final de la cola para procesarla más adelante."""
    self.queueu.enqueue(registro)

## 2. Clase `DataProcessor`

Implemente:

```python
class DataProcessor:
    ...
```

Debe utilizar:

- un `ArrayQueue` para los registros pendientes;
- un `ArrayStack` para el historial de cambios;
- un `list` para mantener el estado actual.

### Restricción

No sustituya `ArrayQueue` o `ArrayStack` por `list`, `collections.deque` u otra estructura para realizar las funciones que corresponden a la Queue o al Stack.

La clase `DataProcessor` no debe imprimir resultados. Los métodos deben devolver los valores especificados. Las impresiones utilizadas para demostrar el funcionamiento deben realizarse en las celdas de prueba.

In [ ]:
class DataProcessor():
    def __init__(self):
        self.queue = ArrayQueue()
        self.stack = ArrayStack()
        self.lista = []

## 3. `add(record)`

Agrega un registro a la `ArrayQueue`.

```python
processor.add(("S01", "temperature", 23.5))
```

Requisitos:

- agrega el registro a la cola;
- **no procesa** el registro;
- conserva el orden de llegada.

No se requiere un valor de retorno.

Debe rechazar registros que no tengan exactamente tres componentes o cuyo `value` no sea numérico. El tipo concreto de excepción para estos errores puede ser elegido por el estudiante, pero debe documentarse y utilizarse consistentemente.

In [30]:
def add(self, record):

    """Agrega un registro a la cola. Valida el formato y el tipo de dato. Si pasa los filtros, se encola. """

    if type(record) != tuple:
        raise ValueError("El registro debe ser una tupla")
    
    if len(record) != 3:
        raise ValueError("El registro debe tener exactamente 3 elementos")

    sensor, variable, value = record

    if type(value) != int and type(value) != float: 
        raise TypeError("El valor debe ser un número")

    self.queue.enqueue(record)

## 4. `process_next()`

Procesa el siguiente registro pendiente.

Debe:

1. obtener el siguiente registro de la Queue;
2. procesarlo;
3. actualizar el estado actual;
4. guardar en el Stack la información necesaria para poder deshacer exactamente ese cambio.

### FIFO

Si se ejecuta:

```python
add(A)
add(B)
add(C)
```

las llamadas sucesivas a `process_next()` deben devolver/procesar `A`, luego `B` y luego `C`.

### Actualización

Si se procesa:

```python
("S01", "temperature", 23.5)
```

en el estado se debe reflejar:

```python
('S01', 'temperature', 23.5)
```

Si después se procesa `("S01", "temperature", 27.0)`, el valor actual debe ser `27.0`.

### Historial

Se debe agregar al `Stack` respectivo

### Retorno

Debe devolver el registro que acaba de ser procesado.

### Queue vacía

Si no hay registros pendientes, debe producir `Empty, el error creado en el repositorio `Goodrich`

In [31]:
from goodrich.exceptions import Empty

In [32]:
def process_next(self):
    
    """ Saca el siguiente registro de la cola, lo actualiza en la lista y guarda el historial."""
    
    if self.queue.is_empty():
        raise Empty("No hay registros pendientes")

    record = self.queue.dequeue()
    
    sensor, variable, value = record

    encontrado = False

    for i in range(len(self.lista)):
        registro_actual = self.lista[i]

        if registro_actual[0] == sensor and registro_actual[1] == variable:

            valor_viejo = registro_actual[2]

            self.stack.push(("Actualizar", sensor, variable, valor_viejo))

            self.lista[i] = record

            encontrado = True

            break

    if encontrado == False:
        self.stack.push(("NUEVO", sensor, variable, None))

        self.lista.append(record)
        
    return record

## 5. `undo()`

Deshace el último cambio realizado mediante `process_next()`.

Ejemplo:

```text
20 → 25 → 30
```

Después de un `undo()`:

```text
20 → 25
```

Después de otro:

```text
20
```

Los cambios deben deshacerse en orden LIFO.

### Dato creado por primera vez

Suponga que inicialmente no existe `('S01', 'temperature', x)`, después de procesar:

```python
("S01", "temperature", 23.5)
```

el dato existe. Si se ejecuta `undo()`, debe volver a **no existir**.

### Historial vacío

Si no hay cambios que deshacer, debe producir `Empty`.

No se requiere un valor de retorno.

In [33]:
def undo(self):
    
    """Deshace el último cambio realizado en la lista. Los cambios se deshacen en orden LIFO gracias a la pila (Stack)."""

    if self.stack.is_empty():
        raise Empty("No hay cambios para deshacer.")

    accion_guardada = self.stack.pop()
    tipo_accion, sensor, variable, valor_viejo = accion_guardada

    for i in range(len(self.lista)):
        registro_actual = self.lista[i]

        if registro_actual[0] == sensor and registro_actual[1] == variable:

            if tipo_accion == "Nuevo":
                self.lista.pop(i)

            elif tipo_accion == "Actualizar":
                self.lista[i] = (sensor, variable, valor_viejo)

            break
    

## 6. `pending()`

Devuelve el número de registros que todavía esperan ser procesados.

Por ejemplo, después de:

```python
add(A)
add(B)
add(C)
```

`pending()` debe devolver `3`. Después de `process_next()`, debe devolver `2`.

In [34]:
def pending(self):
    
    """Devuelve el número de registros que todavía esperan ser procesados."""

    return len(self.queue)

## 7. `current_value(sensor, variable)`

Devuelve el valor actual asociado con una combinación de sensor y variable.

Ejemplo:

```python
current_value("S01", "temperature")
```

puede devolver `23.5`.

Si nunca se ha procesado un registro para esa combinación, debe producir `KeyError`.

In [35]:
def current_value(self, sensor, variable):
        indice = self._buscar(sensor, variable)
        if indice is None:
            raise KeyError(f"No existe registro para ({sensor}, {variable}).")
        return self.lista[indice][2]

# 8. Ejemplo completo

Considere:

```python
A = ("S01", "temperature", 20)
B = ("S01", "temperature", 25)
C = ("S01", "humidity", 60)
```

Después de `add(A)`, `add(B)`, `add(C)`, la Queue contiene `A → B → C`.

Después de procesar A, el estado contiene:

```text
S01 / temperature → 20
```

Después de procesar B:

```text
S01 / temperature → 25
```

Después de procesar C:

```text
S01 / temperature → 25
S01 / humidity    → 60
```

Un `undo()` elimina el efecto de C. Otro `undo()` elimina el efecto de B. Otro `undo()` elimina el efecto de A y el estado vuelve a estar vacío.

# 9. Pruebas obligatorias

Incluya pruebas para, como mínimo:

1. `pending()` sobre un procesador vacío.
2. Agregar un registro.
3. Agregar varios registros.
4. Verificar procesamiento FIFO.
5. Procesar un registro.
6. Procesar varios registros.
7. Actualizar una variable existente.
8. Consultar el valor actual.
9. Realizar un `undo()`.
10. Realizar varios `undo()` consecutivos.
11. Procesar cuando la Queue está vacía.
12. Hacer `undo()` cuando el historial está vacío.
13. Deshacer la creación de un dato que antes no existía.
14. Hacer varios cambios sobre la misma variable.
15. Agregar un registro con formato incorrecto.
16. Agregar un registro cuyo valor no sea numérico.
17. Consultar un sensor/variable que nunca haya sido procesado.

In [36]:
"""1. pending() sobre un procesador vacío"""
p = DataProcessor()
assert p.pending() == 0, "Un procesador recién creado debe tener 0 pendientes"
print("Prueba 1 OK: pending() == 0 en procesador vacío")

AttributeError: 'DataProcessor' object has no attribute 'pending'

In [ ]:
"""2. Agregar un registro"""
p = DataProcessor()
p.add(("S01", "temperature", 23.5))
assert p.pending() == 1, "Tras agregar 1 registro, pending() debe ser 1"
print("Prueba 2 OK: agregar un registro incrementa pending() a 1")

In [ ]:
"""3. Agregar varios registros"""
p = DataProcessor()
p.add(("S01", "temperature", 20))
p.add(("S01", "temperature", 25))
p.add(("S01", "humidity", 60))
assert p.pending() == 3, "Tras agregar 3 registros, pending() debe ser 3"
print("Prueba 3 OK: agregar varios registros acumula pendientes correctamente")

In [ ]:
"""4. Verificar procesamiento FIFO"""
p = DataProcessor()
A = ("S01", "temperature", 20)
B = ("S01", "temperature", 25)
C = ("S01", "humidity", 60)
p.add(A); p.add(B); p.add(C)

r1 = p.process_next()
r2 = p.process_next()
r3 = p.process_next()

assert r1 == A, f"El primer procesado debe ser A, se obtuvo {r1}"
assert r2 == B, f"El segundo procesado debe ser B, se obtuvo {r2}"
assert r3 == C, f"El tercer procesado debe ser C, se obtuvo {r3}"
print("Prueba 4 OK: procesamiento FIFO respetado (A → B → C)")

In [ ]:
"""5. Procesar un registro"""
p = DataProcessor()
p.add(("S02", "temperature", 30))
resultado = p.process_next()

assert resultado == ("S02", "temperature", 30), "process_next debe devolver el registro procesado"
assert p.pending() == 0, "Tras procesar, no deben quedar pendientes"
assert p.current_value("S02", "temperature") == 30, "El estado debe reflejar el valor procesado"
print("Prueba 5 OK: procesar un registro actualiza estado y devuelve el registro")

In [ ]:
"""6. Procesar varios registros"""
p = DataProcessor()
p.add(("A", "x", 1))
p.add(("B", "y", 2))
p.add(("C", "z", 3))

p.process_next()
p.process_next()
p.process_next()

assert p.current_value("A", "x") == 1
assert p.current_value("B", "y") == 2
assert p.current_value("C", "z") == 3
assert p.pending() == 0
print("Prueba 6 OK: procesar varios registros refleja todos los cambios")

In [ ]:
"""7. Actualizar una variable existente"""
p = DataProcessor()
p.add(("S01", "temperature", 20))
p.process_next()
p.add(("S01", "temperature", 27))
p.process_next()

assert p.current_value("S01", "temperature") == 27, "El valor debe haberse actualizado a 27"
print("Prueba 7 OK: actualizar una variable existente reemplaza el valor")

In [ ]:
"""8. Consultar el valor actual"""
p = DataProcessor()
p.add(("S01", "temperature", 23.5))
p.process_next()

valor = p.current_value("S01", "temperature")
assert valor == 23.5, f"current_value debe devolver 23.5, se obtuvo {valor}"
print("Prueba 8 OK: current_value devuelve el valor correcto")

In [ ]:
"""9. Realizar un undo()"""
p = DataProcessor()
p.add(("S01", "temperature", 20))
p.process_next()
p.add(("S01", "temperature", 25))
p.process_next()

p.undo()
assert p.current_value("S01", "temperature") == 20, "Tras undo, el valor debe volver a 20"
print("Prueba 9 OK: un undo revierte el último cambio")

In [ ]:
"""10. Varios undo() consecutivos"""
p = DataProcessor()
for v in [20, 25, 30]:
    p.add(("S01", "temperature", v))
    p.process_next()

p.undo()
assert p.current_value("S01", "temperature") == 25, "Primer undo debe dejar 25"
p.undo()
assert p.current_value("S01", "temperature") == 20, "Segundo undo debe dejar 20"
p.undo()
try:
    p.current_value("S01", "temperature")
    raise AssertionError("Tras el tercer undo, el registro no debe existir")
except KeyError:
    pass
print("Prueba 10 OK: varios undo() revierten en orden LIFO hasta vaciar")

In [ ]:
"""11. Procesar cuando la Queue está vacía"""
p = DataProcessor()
try:
    p.process_next()
    raise AssertionError("process_next debe lanzar Empty si no hay pendientes")
except Empty:
    print("Prueba 11 OK: process_next lanza Empty con cola vacía")

In [ ]:
""" 12. Hacer undo() cuando el historial está vacío"""
p = DataProcessor()
try:
    p.undo()
    raise AssertionError("undo debe lanzar Empty si no hay historial")
except Empty:
    print("Prueba 12 OK: undo lanza Empty con historial vacío")

In [ ]:
"""13. Deshacer la creación de un dato que antes no existía"""
p = DataProcessor()
p.add(("S99", "pressure", 101.3))
p.process_next()

# El dato existe
assert p.current_value("S99", "pressure") == 101.3

p.undo()

# Tras el undo, el dato NO debe existir
try:
    p.current_value("S99", "pressure")
    raise AssertionError("El dato debe haber dejado de existir tras el undo")
except KeyError:
    print("Prueba 13 OK: undo elimina un dato creado por primera vez")

In [ ]:
"""14. Hacer varios cambios sobre la misma variable"""
p = DataProcessor()
for v in [20, 25, 30]:
    p.add(("S01", "temperature", v))
    p.process_next()

assert p.current_value("S01", "temperature") == 30

p.undo()
assert p.current_value("S01", "temperature") == 25, "Tras 1er undo debe quedar 25"
p.undo()
assert p.current_value("S01", "temperature") == 20, "Tras 2do undo debe quedar 20"
print("Prueba 14 OK: varios cambios sobre la misma variable se deshacen correctamente")

In [ ]:
"""15. Agregar un registro con formato incorrecto"""
p = DataProcessor()

# 15a. No es tupla
try:
    p.add(["S01", "temperature", 20])
    raise AssertionError("Debe lanzar ValueError si no es tupla")
except ValueError:
    pass

# 15b. Tupla con menos de 3 elementos
try:
    p.add(("S01", "temperature"))
    raise AssertionError("Debe lanzar ValueError si no tiene 3 elementos")
except ValueError:
    pass

# 15c. Tupla con más de 3 elementos
try:
    p.add(("S01", "temperature", 20, "extra"))
    raise AssertionError("Debe lanzar ValueError si tiene más de 3 elementos")
except ValueError:
    pass

print("Prueba 15 OK: registros con formato incorrecto lanzan ValueError")

In [ ]:
"""16. Agregar un registro cuyo valor no sea numérico"""
p = DataProcessor()

# 16a. String
try:
    p.add(("S01", "temperature", "caliente"))
    raise AssertionError("Debe lanzar TypeError si el valor es string")
except TypeError:
    pass

# 16b. None
try:
    p.add(("S01", "temperature", None))
    raise AssertionError("Debe lanzar TypeError si el valor es None")
except TypeError:
    pass

# 16c. Lista
try:
    p.add(("S01", "temperature", [1, 2]))
    raise AssertionError("Debe lanzar TypeError si el valor es lista")
except TypeError:
    pass

print("Prueba 16 OK: valores no numéricos lanzan TypeError")

In [ ]:
"""17. Consultar un sensor/variable que nunca haya sido procesado"""
p = DataProcessor()
p.add(("S01", "temperature", 20))
p.process_next()

# Esta combinación sí existe
assert p.current_value("S01", "temperature") == 20

# Esta combinación nunca se procesó
try:
    p.current_value("XX", "yy")
    raise AssertionError("Debe lanzar KeyError para sensor/variable inexistente")
except KeyError:
    print("Prueba 17 OK: current_value lanza KeyError si la combinación no existe")

In [ ]:
print("=" * 50)
print("TODAS LAS PRUEBAS PASARON (1–17)")
print("=" * 50)

# 10. Análisis de complejidad

Explique la complejidad temporal de:

- `add`
- `process_next`
- `undo`
- `pending`
- `current_value`

Justifique qué operaciones determinan cada complejidad e indique qué estructuras auxiliares utiliza el sistema.

# 11. Restricciones

1. Utilice las clases `ArrayStack` y `ArrayQueue` proporcionadas.
2. No las reemplace por `list`, `deque` u otra estructura equivalente.
3. No modifique las implementaciones proporcionadas.
4. La implementación debe estar contenida en `DataProcessor`.
5. Incluya las pruebas solicitadas.
6. Explique brevemente sus decisiones de diseño.

# 12. Bonus — `redo()`

Como extensión opcional, implemente:

```python
redo()
```

Después de un `undo()`, el sistema debe poder volver a aplicar el cambio que acaba de deshacerse.

Por ejemplo:

```text
20 → 25 → 30
undo()  → 20 → 25
redo()  → 20 → 25 → 30
```

El estudiante debe explicar qué estructuras utiliza para implementar `redo()` y por qué. No se proporciona la estrategia de implementación.


# 13. Entrega

La entrega debe contener:

- implementación completa de `DataProcessor`;
- pruebas solicitadas;
- explicación breve de las decisiones de diseño;
- análisis de complejidad temporal;
- si realiza el bonus, implementación y explicación de `redo()`.